# Subfase 7 — Modelado cuántico h5 (Notebook autocontenido)

Notebook dedicado a flujo cuántico (PCA<=5 + FidelityQuantumKernel + SVC en statevector)

In [ ]:
from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from bvg_core.config import (  
  COMPANIES, GLOBAL_SEED, TEST_SIZE, DATA_MASTER_PATH as DATASET_PATH, 
  QUANTUM_DIR, FECHA_COL_2, EMPRESA_COL, FEATURE_EXCLUDE, TARGET_COL
)
from bvg_core.utils import safe_company, sha256_file
from bvg_core.splits import temporal_split_company

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC

from qiskit.primitives import StatevectorSampler
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.state_fidelities import ComputeUncompute
import qiskit
import qiskit_machine_learning


In [ ]:
np.random.seed(GLOBAL_SEED)

QUANTUM_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Dataset no encontrado: {DATASET_PATH}')

df = pd.read_csv(DATASET_PATH)

required = {FECHA_COL_2, EMPRESA_COL, TARGET_COL}
missing = sorted(required.difference(df.columns))
if missing:
    raise ValueError(f'Columnas faltantes: {missing}')

feature_cols = [c for c in df.columns if c not in FEATURE_EXCLUDE]
if not feature_cols:
    raise ValueError('No se pudieron inferir feature_cols.')

df = df.assign(**{FECHA_COL_2: pd.to_datetime(df[FECHA_COL_2], errors='coerce')})
before = len(df)
df = df.dropna(subset=feature_cols + [TARGET_COL, FECHA_COL_2, EMPRESA_COL]).copy()
after = len(df)
if df.empty:
    raise ValueError('Dataset vacío tras limpieza de NaN.')

df = df.sort_values([EMPRESA_COL, FECHA_COL_2], kind='stable').reset_index(drop=True)
quality = {
    'total_rows': int(before),
    'after_nan_drop_rows': int(after),
    'dropped_nan_rows': int(before - after),
}
quality

{'total_rows': 2841, 'after_nan_drop_rows': 2831, 'dropped_nan_rows': 10}

In [ ]:
def fit_quantum_svc(X_train: pd.DataFrame, y_train: pd.Series, X_test: pd.DataFrame, y_test: pd.Series) -> dict:
    scaler = RobustScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    pca_n = int(min(5, X_train_s.shape[1]))
    pca = PCA(n_components=pca_n, random_state=GLOBAL_SEED)
    X_train_q = pca.fit_transform(X_train_s)
    X_test_q = pca.transform(X_test_s)

    feature_map = zz_feature_map(feature_dimension=pca_n, reps=2, entanglement='linear')
    fidelity = ComputeUncompute(sampler=StatevectorSampler(seed=GLOBAL_SEED))
    qkernel = FidelityQuantumKernel(feature_map=feature_map, fidelity=fidelity)

    K_train = qkernel.evaluate(x_vec=X_train_q)
    K_test = qkernel.evaluate(x_vec=X_test_q, y_vec=X_train_q)

    svc = SVC(kernel='precomputed', probability=True, class_weight='balanced', random_state=GLOBAL_SEED)
    svc.fit(K_train, y_train)

    y_pred = svc.predict(K_test)

    metrics = {
        'accuracy': float(accuracy_score(y_test, y_pred)),
        'f1': float(f1_score(y_test, y_pred, zero_division=0)),
        'positive_rate_pred': float(np.mean(y_pred)),
        'positive_rate_test': float(np.mean(y_test)),
    }

    return {
        'scaler': scaler,
        'pca': pca,
        'svc': svc,
        'feature_map_config': {
            'name': 'zz_feature_map',
            'feature_dimension': pca_n,
            'reps': 2,
            'entanglement': 'linear',
            'fidelity_backend': 'statevector',
        },
        'X_train_q': X_train_q,
        'X_test_q': X_test_q,
        'metrics': metrics,
    }

def predict_quantum_bundle(bundle: dict, X_raw: pd.DataFrame) -> np.ndarray:
    Xs = bundle['scaler'].transform(X_raw)
    Xq = bundle['pca'].transform(Xs)
    K = bundle['qkernel'].evaluate(x_vec=Xq, y_vec=bundle['X_train_q'])
    return bundle['svc'].predict(K)

def export_and_validate_quantum(company: str, tr: pd.DataFrame, te: pd.DataFrame, X_test: pd.DataFrame, trained: dict) -> dict:
    tag = safe_company(company)
    prefix = f'{tag}_h5'

    scaler_path = QUANTUM_DIR / f'{prefix}_scaler.joblib'
    pca_path = QUANTUM_DIR / f'{prefix}_pca.joblib'
    svc_path = QUANTUM_DIR / f'{prefix}_svc.joblib'
    cfg_path = QUANTUM_DIR / f'{prefix}_kernel_config.json'
    manifest_path = QUANTUM_DIR / f'{prefix}_manifest.json'

    joblib.dump(trained['scaler'], scaler_path)
    joblib.dump(trained['pca'], pca_path)
    joblib.dump(trained['svc'], svc_path)
    cfg_path.write_text(json.dumps(trained['feature_map_config'], ensure_ascii=False, indent=2), encoding='utf-8')

    checksums = {
        scaler_path.name: sha256_file(scaler_path),
        pca_path.name: sha256_file(pca_path),
        svc_path.name: sha256_file(svc_path),
        cfg_path.name: sha256_file(cfg_path),
    }

    manifest = {
        'project': 'tesis_main',
        'subphase': 'subfase-7-modelado-cuantico-h5',
        'company': company,
        'horizonte': 'h5',
        'model_family': 'quantum',
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'random_seed': GLOBAL_SEED,
        'train_start_date': str(tr[FECHA_COL_2].min().date()),
        'train_end_date': str(tr[FECHA_COL_2].max().date()),
        'feature_columns': list(feature_cols),
        'target_column': TARGET_COL,
        'sklearn_version': sklearn.__version__,
        'qiskit_version': qiskit.__version__,
        'qiskit_ml_version': qiskit_machine_learning.__version__,
        'params_fixed': {'svc_kernel': 'precomputed', 'probability': True, 'class_weight': 'balanced'},
        'pca_n_components': int(trained['pca'].n_components_),
        'artifact_checksums_sha256': checksums,
        'data_snapshot_hash': hashlib.sha256(json.dumps({
            'company': company,
            'train_start_date': str(tr[FECHA_COL_2].min().date()),
            'train_end_date': str(tr[FECHA_COL_2].max().date()),
            'n_train': int(len(tr)),
            'feature_columns': list(feature_cols),
        }, sort_keys=True).encode('utf-8')).hexdigest(),
        'metrics': trained['metrics'],
    }
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

    scaler_r = joblib.load(scaler_path)
    pca_r = joblib.load(pca_path)
    svc_r = joblib.load(svc_path)

    feature_map = zz_feature_map(
        feature_dimension=int(trained['feature_map_config']['feature_dimension']),
        reps=int(trained['feature_map_config']['reps']),
        entanglement=trained['feature_map_config']['entanglement'],
    )
    fidelity = ComputeUncompute(sampler=StatevectorSampler(seed=GLOBAL_SEED))
    qkernel_r = FidelityQuantumKernel(feature_map=feature_map, fidelity=fidelity)

    X_train_q_r = pca_r.transform(scaler_r.transform(tr.loc[:, feature_cols].copy()))
    X_smoke_q = pca_r.transform(scaler_r.transform(X_test.iloc[:5].copy()))
    K_smoke = qkernel_r.evaluate(x_vec=X_smoke_q, y_vec=X_train_q_r)
    smoke = [int(v) for v in svc_r.predict(K_smoke)]

    return {
        'artifact_paths': {
            'scaler_path': str(scaler_path),
            'pca_path': str(pca_path),
            'svc_path': str(svc_path),
            'kernel_config_path': str(cfg_path),
            'manifest_path': str(manifest_path),
        },
        'checksums_ok': all(sha256_file(QUANTUM_DIR / fname) == digest for fname, digest in checksums.items()),
        'smoke_predictions': smoke,
    }

In [ ]:
run = {
    'seed': GLOBAL_SEED,
    'dataset_path': str(DATASET_PATH),
    'quality': quality,
    'feature_columns': feature_cols,
    'companies': {},
}

for company in COMPANIES:
    tr, te, X_train, y_train, X_test, y_test = temporal_split_company(
        df,
        company,
        test_size=TEST_SIZE,
        company_col=EMPRESA_COL,
        date_col=FECHA_COL_2,
        target_col=TARGET_COL,
        feature_cols=feature_cols,
    )
    trained = fit_quantum_svc(X_train, y_train, X_test, y_test)

    # Se reconstruye qkernel para predicción in-memory y trazabilidad
    fm = zz_feature_map(
        feature_dimension=int(trained['feature_map_config']['feature_dimension']),
        reps=int(trained['feature_map_config']['reps']),
        entanglement=trained['feature_map_config']['entanglement'],
    )
    fidelity = ComputeUncompute(sampler=StatevectorSampler(seed=GLOBAL_SEED))
    qkernel = FidelityQuantumKernel(feature_map=fm, fidelity=fidelity)

    trained_bundle = dict(trained)
    trained_bundle['qkernel'] = qkernel
    exported = export_and_validate_quantum(company, tr, te, X_test, trained_bundle)

    run['companies'][company] = {
        'metrics': trained['metrics'],
        'pca_n_components': int(trained['pca'].n_components_),
        'feature_map_config': trained['feature_map_config'],
        'export': exported,
    }

summary_path = QUANTUM_DIR / 'h5_subfase7_run_summary.json'
summary_path.write_text(json.dumps(run, ensure_ascii=False, indent=2), encoding='utf-8')
run

C:\Users\leynd\AppData\Local\Temp\ipykernel_13300\757604991.py:27: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=pca_n, reps=2, entanglement='linear')
C:\Users\leynd\AppData\Local\Temp\ipykernel_13300\2173688756.py:14: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  fm = ZZFeatureMap(
C:\Users\leynd\AppData\Local\Temp\ipykernel_13300\757604991.py:123: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._

{'seed': 42,
 'dataset_path': 'C:\\Users\\leynd\\OneDrive\\Escritorio\\Tesis\\implementaciones\\desarrollov3\\data\\processed\\BVG_features_svc_master.csv',
 'quality': {'total_rows': 2841,
  'after_nan_drop_rows': 2831,
  'dropped_nan_rows': 10},
 'feature_columns': ['close_last',
  'close_vwap',
  'volume_shares_day',
  'turnover_value_day',
  'n_trades_day',
  'ret_lag_1',
  'ret_lag_2',
  'ret_lag_3',
  'mom_3',
  'mom_5',
  'mom_10',
  'vol_5',
  'vol_10',
  'regime_vol_ratio',
  'ma_5',
  'ma_10',
  'ma_gap',
  'price_vs_ma10',
  'rsi_14',
  'turnover_log1p',
  'volume_log1p',
  'avg_trade_size_log1p',
  'amihud_5',
  'days_since_trade'],
 'companies': {'BANCO GUAYAQUIL S.A.': {'metrics': {'accuracy': 0.36666666666666664,
    'f1': 0.45714285714285713,
    'positive_rate_pred': 0.3333333333333333,
    'positive_rate_test': 0.8333333333333334},
   'pca_n_components': 5,
   'feature_map_config': {'name': 'ZZFeatureMap',
    'feature_dimension': 5,
    'reps': 2,
    'entanglement':